# AI Code Auditor v2 — QLoRA Fine-tuning
**Model:** DeepSeek-Coder-6.7B | **Method:** QLoRA | **Dataset:** Big-Vul v2 (top-10 CWEs)

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q transformers==4.40.2 peft==0.10.0 trl==0.8.6 bitsandbytes==0.45.3 accelerate==0.29.3 datasets==2.19.1
print('Done')

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
print('Environment set')

In [ ]:
import os, json
TRAIN_PATH = None
VAL_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        if f == 'train.jsonl': TRAIN_PATH = full
        if f == 'val.jsonl': VAL_PATH = full
assert TRAIN_PATH and VAL_PATH, 'Dataset not found'
print(f'Train: {TRAIN_PATH}')
print(f'Val:   {VAL_PATH}')
with open(TRAIN_PATH) as f:
    sample = json.loads(f.readline())
print(f'CWE: {sample["cwe"]} | Completion preview:')
print(sample['completion'][:200])

In [ ]:
BASE_MODEL = 'deepseek-ai/deepseek-coder-6.7b-base'
OUTPUT_DIR = '/kaggle/working/lora_adapter'
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-4
MAX_SEQ_LEN = 512
SEED = 42
print(f'Model: {BASE_MODEL}')
print('Config ready')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f'CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.use_cache = False
print(f'Loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')
val_dataset   = load_dataset('json', data_files=VAL_PATH,   split='train')
print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')
print(train_dataset[0]['text'][:300])

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import json, shutil, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=25,
    evaluation_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    optim='paged_adamw_32bit',
    group_by_length=True,
    report_to='none',
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f'Train samples  : {len(train_dataset):,}')
print(f'Steps per epoch: {steps_per_epoch}')
print(f'Total steps    : {steps_per_epoch * NUM_EPOCHS}')

trainer.train()

# Save adapter
out = Path(OUTPUT_DIR)
trainer.model.save_pretrained(out)
tokenizer.save_pretrained(out)
print(f'Adapter saved: {[f.name for f in out.iterdir()]}')

# Save log + plot
log = trainer.state.log_history
with open('/kaggle/working/training_log.json', 'w') as f:
    json.dump(log, f, indent=2)

tl = [(e['step'], e['loss']) for e in log if 'loss' in e and 'eval_loss' not in e]
el = [(e['step'], e['eval_loss']) for e in log if 'eval_loss' in e]
fig, ax = plt.subplots(figsize=(10, 4))
if tl:
    s, l = zip(*tl)
    ax.plot(s, l, label='Train', color='steelblue', alpha=0.7)
if el:
    s, l = zip(*el)
    ax.plot(s, l, label='Eval', color='coral', linestyle='--', marker='o')
ax.set_title('DeepSeek-Coder-6.7B QLoRA on Big-Vul v2', fontweight='bold')
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/training_loss.png', dpi=150)
plt.close()

# Zip
shutil.make_archive('/kaggle/working/lora_adapter_download', 'zip', str(out))
print(f'Zip: {Path("/kaggle/working/lora_adapter_download.zip").stat().st_size/1e6:.0f} MB')
print('DONE — Click Save Version now!')

In [ ]:
from IPython.display import FileLink, display
display(FileLink('lora_adapter_download.zip'))
display(FileLink('training_log.json'))
display(FileLink('training_loss.png'))